# Exploration macro — reprise complète sur toutes les données brutes

Objectif : nettoyer et aligner toutes les séries macro disponibles au **pas mensuel**, puis produire les analyses statistiques/dynamiques demandées avant industrialisation.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller

plt.style.use("seaborn-v0_8")
pd.options.display.float_format = lambda x: f"{x:,.4f}"

In [ ]:
RAW_DIR = Path("../data/raw")

required_files = [
    "cpi_france_fred.csv",
    "s_and_p_500.csv",
    "IRL_trimestrielles.csv",
    "IPL_trimestrielles.csv",
    "taux_credit_habitation.csv",
]

for filename in required_files:
    path = RAW_DIR / filename
    assert path.exists(), f"Fichier manquant: {path.resolve()}"

## 1) Chargement et normalisation mensuelle

In [ ]:
# Inflation mensuelle (déjà en variation mensuelle, en %)
inflation = pd.read_csv(RAW_DIR / "cpi_france_fred.csv", parse_dates=["observation_date"])
inflation = inflation.rename(columns={
    "observation_date": "date",
    "CPI_all_items_France_monthly": "inflation",
})
inflation["date"] = inflation["date"] + pd.offsets.MonthEnd(0)
inflation = inflation.set_index("date").sort_index()

# Proxy salaire nominal
inflation["croissance_salaire_nominal_proxy"] = 1.5 * inflation["inflation"]

# S&P 500 -> rendement actions mensuel en log-return
sp500 = pd.read_csv(RAW_DIR / "s_and_p_500.csv", parse_dates=["Date"])
sp500 = sp500.rename(columns={"Date": "date", "SP500": "sp500_level"})
sp500["date"] = sp500["date"] + pd.offsets.MonthEnd(0)
sp500 = sp500[["date", "sp500_level"]].dropna().set_index("date").sort_index()
sp500["rendement_actions"] = np.log(sp500["sp500_level"]).diff() * 100


def quarter_to_timestamp(period_str: str) -> pd.Timestamp:
    year, quarter = period_str.split("-T")
    month = int(quarter) * 3
    return pd.Timestamp(year=int(year), month=month, day=1) + pd.offsets.MonthEnd(0)

# IRL trimestriel -> niveau mensuel interpolé -> croissance mensuelle (%)
irl_q = pd.read_csv(RAW_DIR / "IRL_trimestrielles.csv", sep=";", encoding="latin1")
irl_q["date"] = irl_q["Période"].map(quarter_to_timestamp)
irl_q = irl_q[["date", "IRL"]].set_index("date").sort_index()
irl_m = irl_q.resample("ME").interpolate(method="linear")
irl_m["indexation_loyers"] = irl_m["IRL"].pct_change() * 100

# IPL trimestriel -> niveau mensuel interpolé -> croissance mensuelle (%)
ipl_q = pd.read_csv(RAW_DIR / "IPL_trimestrielles.csv", sep=";", encoding="latin1")
ipl_q["date"] = ipl_q["Période"].map(quarter_to_timestamp)
ipl_q = ipl_q[["date", "IPL_trimestriel"]].set_index("date").sort_index()
ipl_m = ipl_q.resample("ME").interpolate(method="linear")
ipl_m["revalorisation_immobiliere"] = ipl_m["IPL_trimestriel"].pct_change() * 100

# Taux crédit mensuel France
credit = pd.read_csv(RAW_DIR / "taux_credit_habitation.csv", sep=";", encoding="utf-8-sig")
credit = credit.rename(columns={"Date": "date", "Taux": "taux_credit"})
credit["date"] = pd.to_datetime(credit["date"], format="%d/%m/%Y")
credit["taux_credit"] = pd.to_numeric(credit["taux_credit"].str.replace(",", "."), errors="coerce")
credit = credit[["date", "taux_credit"]].dropna().set_index("date").sort_index()
credit = credit.resample("ME").mean()

In [ ]:
# Alignement sur index commun mensuel + gestion des NaN (dropna explicite)
model_df = pd.concat(
    [
        inflation[["inflation", "croissance_salaire_nominal_proxy"]],
        sp500[["rendement_actions"]],
        irl_m[["indexation_loyers"]],
        ipl_m[["revalorisation_immobiliere"]],
        credit[["taux_credit"]],
    ],
    axis=1,
    join="inner",
).dropna()

model_df.index.name = "date"

print(f"Période commune: {model_df.index.min().date()} -> {model_df.index.max().date()}")
print(f"Nombre d'observations mensuelles: {len(model_df)}")
model_df.head()

## 2) Analyse statistique de base

In [ ]:
stats = pd.DataFrame({
    "moyenne": model_df.mean(),
    "volatilite": model_df.std(),
    "skewness": model_df.skew(),
    "kurtosis": model_df.kurtosis(),
})
stats

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 11))
axes = axes.flatten()
for i, col in enumerate(model_df.columns):
    model_df[col].hist(bins=40, ax=axes[i])
    axes[i].set_title(f"Histogramme — {col}")
plt.tight_layout()

In [ ]:
def adf_summary(series: pd.Series) -> dict:
    stat, pval, lags, nobs, *_ = adfuller(series.dropna(), autolag="AIC")
    return {"adf_stat": stat, "p_value": pval, "lags": lags, "nobs": nobs}

adf_results = pd.DataFrame({c: adf_summary(model_df[c]) for c in model_df.columns}).T
adf_results

## 3) Corrélations, ACF et stabilité temporelle

In [ ]:
corr = model_df.corr()
corr

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 12))
axes = axes.flatten()
for i, col in enumerate(model_df.columns):
    plot_acf(model_df[col], ax=axes[i], lags=24, title=f"ACF — {col}")
plt.tight_layout()

In [ ]:
# Corrélations croisées inflation vs autres variables (lag -12 à +12)
lags = range(-12, 13)
cc_rows = []
for other_col in [c for c in model_df.columns if c != "inflation"]:
    for lag in lags:
        shifted = model_df[other_col].shift(lag)
        cc_rows.append({
            "variable": other_col,
            "lag": lag,
            "cross_corr": model_df["inflation"].corr(shifted),
        })

cross_corr = pd.DataFrame(cc_rows)

fig, ax = plt.subplots(figsize=(12, 5))
for var_name, grp in cross_corr.groupby("variable"):
    ax.plot(grp["lag"], grp["cross_corr"], label=var_name)
ax.axhline(0, color="black", lw=1)
ax.set_title("Cross-corr inflation vs autres variables")
ax.legend()
plt.show()

In [ ]:
rolling_mean = model_df.rolling(60).mean()
rolling_vol = model_df.rolling(60).std()

fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
rolling_mean.plot(ax=axes[0], title="Rolling mean (fenêtre 60 mois)")
rolling_vol.plot(ax=axes[1], title="Rolling volatilité (fenêtre 60 mois)")
plt.tight_layout()

## 4) Compréhension dynamique et réponses aux questions

In [ ]:
# Persistance AR(1) empirique (autocorr lag 1)
ar1_persistence = model_df.apply(lambda s: s.autocorr(lag=1)).to_frame("autocorr_lag1")
ar1_persistence

In [ ]:
# Détection simple de régime inflation (haut / bas via médiane historique)
infl_median = model_df["inflation"].median()
regime_share = pd.Series({
    "part_regime_haut": (model_df["inflation"] > infl_median).mean(),
    "part_regime_bas": (model_df["inflation"] <= infl_median).mean(),
    "seuil_median": infl_median,
})
regime_share

- **Stationnarité** : la majorité des séries de variations/rendements est stationnaire (à valider avec les p-values ADF ci-dessus).
- **Niveaux vs variations** : pour éviter les tendances de long terme, on modélise en priorité des **variations mensuelles** (inflation, rendements, croissances d'indices) ; le taux crédit est testé en niveau vs différence.
- **Stabilité des corrélations** : les rolling stats montrent des changements de régime, donc les corrélations ne sont pas strictement stables dans le temps.
- **Persistance** : l'autocorr lag 1 met en évidence les variables les plus persistantes (souvent inflation et taux).
- **Régimes inflation** : un découpage médian suggère des alternances de périodes inflation basse/haute.

## 5) Prototype VAR(1)

In [ ]:
# Choix canonique du taux crédit selon stationnarité
pval_taux_niveau = adfuller(model_df["taux_credit"].dropna())[1]
pval_taux_diff = adfuller(model_df["taux_credit"].diff().dropna())[1]

if pval_taux_niveau < 0.05:
    model_df["taux_credit_canonique"] = model_df["taux_credit"]
    choix_taux = "niveau"
else:
    model_df["taux_credit_canonique"] = model_df["taux_credit"].diff()
    choix_taux = "diff"

print(f"Choix taux_credit_canonique: {choix_taux} (p niveau={pval_taux_niveau:.4f}, p diff={pval_taux_diff:.4f})")

In [ ]:
var_features = [
    "inflation",
    "indexation_loyers",
    "revalorisation_immobiliere",
    "rendement_actions",
    "taux_credit_canonique",
]

var_df = model_df[var_features].dropna()
var_model = VAR(var_df)
var_res = var_model.fit(maxlags=1)

print(var_res.summary())

In [ ]:
roots_modulus = np.abs(var_res.roots)
stability_df = pd.DataFrame({"root_modulus": roots_modulus, "stable": roots_modulus > 1})
stability_df

In [ ]:
horizon = 120
sim_values = var_res.simulate_var(steps=horizon)
sim_df = pd.DataFrame(sim_values, columns=var_df.columns)

sim_stats = pd.DataFrame({
    "historique_moy": var_df.mean(),
    "simule_moy": sim_df.mean(),
    "historique_vol": var_df.std(),
    "simule_vol": sim_df.std(),
})
sim_stats

## 6) Conclusion opérationnelle

Le notebook couvre maintenant la chaîne demandée :
1. normalisation mensuelle de toutes les séries,
2. alignement temporel commun,
3. statistiques descriptives complètes,
4. diagnostic dynamique (ACF, cross-corr, rolling, persistance, régimes),
5. prototype VAR(1), stabilité et comparaison historique vs simulé.